In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# COMPLETE CORRECTED DRM-Net IMPLEMENTATION
# Paper: "Explainable AI based hybrid DRM-Net transfer learning
#         model for breast cancer detection and classification
#         using ultrasound images"
# Scientific Reports (2025) 15:44170
# Dataset: BUSI (Breast Ultrasound Images Dataset) from Kaggle
# Target: Accuracy=96.71% | AUC=99%
#
# HOW TO RUN ON KAGGLE:
# 1. kaggle.com → New Notebook
# 2. Add Dataset: "aryashah2k/breast-ultrasound-images-dataset"
# 3. Settings → Accelerator → GPU T4 x2
# 4. Paste this code and Run All
# ============================================================

# ─────────────────────────────────────────────
# SECTION 1: INSTALL & IMPORT
# ─────────────────────────────────────────────
import os
import gc
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input, backend as K
from tensorflow.keras.applications import (
    VGG16, EfficientNetB0, InceptionV3,
    DenseNet169, MobileNet, ResNet50
)
from tensorflow.keras.layers import (
    Dense, Flatten, Dropout, GlobalAveragePooling2D,
    Conv2D, MaxPooling2D, BatchNormalization, Concatenate
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from scipy import stats

# GPU config — allow memory growth to avoid OOM
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("✅ Libraries imported")
print(f"TensorFlow: {tf.__version__}")
print(f"GPUs available: {len(gpus)}")

# Mixed precision for speed (T4 GPU supports it)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision: float16 enabled")


# ─────────────────────────────────────────────
# SECTION 2: CONFIG (exact paper values)
# ─────────────────────────────────────────────
IMG_SIZE      = 512       # Resize to 512×512 first (paper Section 3.3)
CROP_SIZE     = 224       # Then center-crop to 224×224 (paper Section 3.3)
BATCH_SIZE    = 32        # Table 4
EPOCHS        = 30        # Table 4
LR            = 0.001     # Table 4
DROPOUT       = 0.3       # Table 4
NUM_CLASSES   = 3

# Augmentation — Table 4
SHEAR         = 0.2
ZOOM          = 0.2
ROTATION      = 24
H_FLIP        = True
V_FLIP        = True

# ImageNet normalization (paper Section 3.3)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Split: 70 / 10 / 20  (Table 10 — best split S3)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
TEST_RATIO  = 0.20

CLASS_NAMES  = ['benign', 'malignant', 'normal']
CLASS_MAP    = {'benign': 0, 'malignant': 1, 'normal': 2}

# ─────────────────────────────────────────────
# SECTION 3: DATASET PATH
# ─────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/datasets/aryashah2k/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT'

# Auto-detect path
for candidate in [
    '/kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT',
    '/kaggle/input/breast-ultrasound-images-dataset',
]:
    if os.path.exists(candidate):
        DATASET_PATH = candidate
        break

print(f"\n📁 Dataset: {DATASET_PATH}")
print("Contents:", os.listdir(DATASET_PATH))


# ─────────────────────────────────────────────
# SECTION 4: IMAGE PREPROCESSING (Section 3.3)
# Resize 512 → Gaussian blur → Sobel → Hist.Eq → Crop 224 → Normalize
# ─────────────────────────────────────────────

def apply_masking(img_uint8):
    """
    Paper equations:
    Eq.1: I_blurred = I * G(sigma)   — Gaussian blur
    Eq.2: E = ∇I                     — Sobel edge detection
    Eq.3: I_equalized = HistEqual(I) — Histogram equalization
    Returns: masked uint8 image (H,W,3)
    """
    # Gaussian blur (Eq.1)
    blurred = cv2.GaussianBlur(img_uint8, (5, 5), 0)

    # Sobel edge map (Eq.2)
    gray    = cv2.cvtColor(blurred, cv2.COLOR_RGB2GRAY)
    sx      = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy      = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edges   = np.sqrt(sx**2 + sy**2)
    edges   = cv2.normalize(edges, None, 0, 255,
                            cv2.NORM_MINMAX).astype(np.uint8)

    # Histogram equalization per channel (Eq.3)
    eq = blurred.copy()
    for c in range(3):
        eq[:, :, c] = cv2.equalizeHist(blurred[:, :, c])

    # Binary mask from edges
    _, mask = cv2.threshold(edges, 30, 255, cv2.THRESH_BINARY)
    mask3   = cv2.merge([mask, mask, mask])
    masked  = cv2.bitwise_and(eq, mask3)
    return masked


def preprocess_image(path):
    """
    Full pipeline per paper Section 3.3:
    1. Load & convert BGR→RGB
    2. Resize to 512×512
    3. Apply masking (Gaussian + Sobel + HistEq)
    4. Center-crop to 224×224
    5. Normalize with ImageNet mean/std
    Returns float32 array (224, 224, 3) or None on error
    """
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Step 1: resize to 512
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

    # Step 2: masking
    img = apply_masking(img)

    # Step 3: center-crop to 224
    s_h = (IMG_SIZE - CROP_SIZE) // 2
    s_w = (IMG_SIZE - CROP_SIZE) // 2
    img = img[s_h:s_h + CROP_SIZE, s_w:s_w + CROP_SIZE]

    # Step 4: normalize
    img = img.astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return img


# ─────────────────────────────────────────────
# SECTION 5: LOAD DATASET
# Paper Table 1: Benign=891, Malignant=421, Normal=266 → 1578 total
# ─────────────────────────────────────────────

def load_dataset(root):
    images, labels = [], []
    for cls_name, cls_idx in CLASS_MAP.items():
        folder = os.path.join(root, cls_name)
        if not os.path.exists(folder):
            print(f"  ⚠️ Missing: {folder}")
            continue
        files = sorted([
            f for f in os.listdir(folder)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            and '_mask' not in f.lower()
        ])
        print(f"  {cls_name}: {len(files)} images")
        for fname in files:
            img = preprocess_image(os.path.join(folder, fname))
            if img is not None:
                images.append(img)
                labels.append(cls_idx)
    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)
    print(f"\n✅ Loaded: {X.shape}")
    print(f"   Benign={np.sum(y==0)}, Malignant={np.sum(y==1)}, Normal={np.sum(y==2)}")
    return X, y


print("\n📊 Loading dataset (takes ~5–10 min with masking)...")
X, y = load_dataset(DATASET_PATH)


# ─────────────────────────────────────────────
# SECTION 6: DATA AUGMENTATION (Table 2 & 4)
# Target: Benign=891, Malignant=842, Normal=798 → 2531 total
# ─────────────────────────────────────────────

def augment_dataset(X, y):
    """
    Augment minority classes to reach paper Table 2 counts.
    De-normalizes before augment, re-normalizes after.
    Uses clipping to avoid float artifacts.
    """
    datagen = ImageDataGenerator(
        shear_range=SHEAR,
        zoom_range=ZOOM,
        rotation_range=ROTATION,
        horizontal_flip=H_FLIP,
        vertical_flip=V_FLIP,
        fill_mode='nearest'
    )

    targets = {0: 891, 1: 842, 2: 798}

    # De-normalize for augmentation pipeline
    X_dn = np.clip(X * STD + MEAN, 0.0, 1.0)

    X_out = list(X)
    y_out = list(y)

    for cls, target in targets.items():
        idx     = np.where(y == cls)[0]
        current = len(idx)
        needed  = target - current
        if needed <= 0:
            print(f"  {CLASS_NAMES[cls]}: {current} (no aug needed)")
            continue
        print(f"  {CLASS_NAMES[cls]}: {current} → {target} (+{needed})")
        pool    = X_dn[idx]
        gen     = datagen.flow(pool, batch_size=1, shuffle=True, seed=42)
        count   = 0
        for batch in gen:
            aug = np.clip(batch[0], 0.0, 1.0)
            aug = (aug - MEAN) / STD          # re-normalize
            X_out.append(aug.astype(np.float32))
            y_out.append(cls)
            count += 1
            if count >= needed:
                break

    X_out = np.array(X_out, dtype=np.float32)
    y_out = np.array(y_out, dtype=np.int32)

    # Shuffle
    perm  = np.random.default_rng(42).permutation(len(X_out))
    X_out = X_out[perm]
    y_out = y_out[perm]

    print(f"\n✅ After augmentation: {X_out.shape}")
    print(f"   Benign={np.sum(y_out==0)}, Malignant={np.sum(y_out==1)}, Normal={np.sum(y_out==2)}")
    return X_out, y_out


print("\n🔄 Augmenting dataset...")
X_aug, y_aug = augment_dataset(X, y)

# Free original array — saves RAM
del X
gc.collect()


# ─────────────────────────────────────────────
# SECTION 7: DATASET SPLIT  70 / 10 / 20
# ─────────────────────────────────────────────

X_tmp,  X_test,  y_tmp,  y_test  = train_test_split(
    X_aug, y_aug,
    test_size=TEST_RATIO, random_state=42, stratify=y_aug
)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)   # 0.125
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=val_frac, random_state=42, stratify=y_tmp
)

print(f"\n📦 Split 70/10/20:")
print(f"   Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

# One-hot encode
y_train_oh = to_categorical(y_train, NUM_CLASSES)
y_val_oh   = to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = to_categorical(y_test,  NUM_CLASSES)


# ─────────────────────────────────────────────
# SECTION 8: BUILD DRM-Net (Paper Figure 7 / Table 3)
#
# KEY FIXES vs original code:
# 1. Input shape = (224, 224, 3)  — backbones take 224×224, NOT 512×512
# 2. training=True in backbone call so BatchNorm stats update correctly
# 3. Conv2D only on DenseNet branch (which has small spatial maps);
#    use GlobalAvgPool for ResNet/MobileNet to avoid huge Flatten tensors
# 4. Each branch: Dense(1024)→Dense(512)→Dense(256)
# 5. Concat → BatchNorm → Dense(512) → Dropout → softmax(3)
# 6. Output dtype cast to float32 (required with mixed_precision)
# ─────────────────────────────────────────────

def build_drm_net(input_shape=(224, 224, 3), unfreeze_last=30):
    """
    DRM-Net: DenseNet169 + ResNet50 + MobileNet
    Architecture faithful to paper Figure 7 & Table 3.

    Each branch:
      backbone → Conv2D 7×7 (1024 filters) → MaxPool 3×3
      → Flatten → Dense(1024) → Dense(512) → Dense(256)

    Note: For DenseNet169 output at 224 input, spatial maps are 7×7.
          For ResNet50 it's also 7×7. For MobileNet it's 7×7.
          Conv2D 7×7 with same padding keeps spatial dims,
          then MaxPool 3×3 reduces them before Flatten.
    """
    inp = Input(shape=input_shape, name='input')

    # ── BRANCH 1: DenseNet169 ──────────────────────────────
    dn_base = DenseNet169(
        weights='imagenet', include_top=False,
        input_shape=input_shape
    )
    # Freeze all except last `unfreeze_last` layers
    for layer in dn_base.layers[:-unfreeze_last]:
        layer.trainable = False
    for layer in dn_base.layers[-unfreeze_last:]:
        layer.trainable = True

    # CRITICAL: training=True so batch norm layers update stats
    d = dn_base(inp, training=True)               # (7,7,1664)
    d = Conv2D(1024, (7, 7), padding='same',
               activation='relu', name='dn_conv')(d)   # (7,7,1024)
    d = MaxPooling2D((3, 3), padding='same',
                     name='dn_pool')(d)                # (3,3,1024) or (4,4,...)
    d = Flatten(name='dn_flat')(d)
    d = Dense(1024, activation='relu', name='dn_d1')(d)
    d = Dropout(DROPOUT)(d)
    d = Dense(512,  activation='relu', name='dn_d2')(d)
    d = Dropout(DROPOUT)(d)
    d = Dense(256,  activation='relu', name='dn_d3')(d)

    # ── BRANCH 2: ResNet50 ────────────────────────────────
    rn_base = ResNet50(
        weights='imagenet', include_top=False,
        input_shape=input_shape
    )
    for layer in rn_base.layers[:-unfreeze_last]:
        layer.trainable = False
    for layer in rn_base.layers[-unfreeze_last:]:
        layer.trainable = True

    r = rn_base(inp, training=True)               # (7,7,2048)
    r = Conv2D(1024, (7, 7), padding='same',
               activation='relu', name='rn_conv')(r)
    r = MaxPooling2D((3, 3), padding='same',
                     name='rn_pool')(r)
    r = Flatten(name='rn_flat')(r)
    r = Dense(1024, activation='relu', name='rn_d1')(r)
    r = Dropout(DROPOUT)(r)
    r = Dense(512,  activation='relu', name='rn_d2')(r)
    r = Dropout(DROPOUT)(r)
    r = Dense(256,  activation='relu', name='rn_d3')(r)

    # ── BRANCH 3: MobileNet ───────────────────────────────
    mn_base = MobileNet(
        weights='imagenet', include_top=False,
        input_shape=input_shape
    )
    for layer in mn_base.layers[:-unfreeze_last]:
        layer.trainable = False
    for layer in mn_base.layers[-unfreeze_last:]:
        layer.trainable = True

    m = mn_base(inp, training=True)               # (7,7,1024)
    m = Conv2D(1024, (7, 7), padding='same',
               activation='relu', name='mn_conv')(m)
    m = MaxPooling2D((3, 3), padding='same',
                     name='mn_pool')(m)
    m = Flatten(name='mn_flat')(m)
    m = Dense(1024, activation='relu', name='mn_d1')(m)
    m = Dropout(DROPOUT)(m)
    m = Dense(512,  activation='relu', name='mn_d2')(m)
    m = Dropout(DROPOUT)(m)
    m = Dense(256,  activation='relu', name='mn_d3')(m)

    # ── CONCAT + FINAL CLASSIFICATION ────────────────────
    merged = Concatenate(name='concat')([d, r, m])   # (768,)
    merged = BatchNormalization(name='bn_merged')(merged)
    merged = Dense(512, activation='relu', name='fc_merged')(merged)
    merged = Dropout(DROPOUT)(merged)

    # Cast to float32 before softmax (mixed_precision requirement)
    out = Dense(NUM_CLASSES, activation='softmax',
                dtype='float32', name='output')(merged)

    model = Model(inputs=inp, outputs=out, name='DRM_Net')
    return model


# ─────────────────────────────────────────────
# SECTION 9: INDIVIDUAL BASELINE TL MODELS (Section 3.4)
# ─────────────────────────────────────────────

def build_baseline(backbone_fn, name, input_shape=(224, 224, 3)):
    """Single TL model with frozen base + custom head"""
    base = backbone_fn(
        weights='imagenet', include_top=False,
        input_shape=input_shape
    )
    base.trainable = False

    inp = Input(shape=input_shape)
    x   = base(inp, training=False)
    x   = GlobalAveragePooling2D()(x)
    x   = BatchNormalization()(x)
    x   = Dense(512,  activation='relu')(x)
    x   = Dropout(DROPOUT)(x)
    x   = Dense(256,  activation='relu')(x)
    x   = Dropout(DROPOUT)(x)
    out = Dense(NUM_CLASSES, activation='softmax',
                dtype='float32')(x)

    model = Model(inp, out, name=name)
    model.compile(
        optimizer=Adam(LR),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model


def get_callbacks(ckpt_path='best_model.h5'):
    return [
        EarlyStopping(monitor='val_accuracy', patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-7, verbose=1),
        ModelCheckpoint(ckpt_path, monitor='val_accuracy',
                        save_best_only=True, verbose=0)
    ]


# ─────────────────────────────────────────────
# SECTION 10: TRAIN BASELINE MODELS
# ─────────────────────────────────────────────

print("\n" + "="*60)
print("TRAINING BASELINE TL MODELS")
print("="*60)

baselines = [
    (VGG16,          'VGG16'),
    (EfficientNetB0, 'EfficientNetB0'),
    (InceptionV3,    'InceptionV3'),
    (DenseNet169,    'DenseNet169'),
    (MobileNet,      'MobileNet'),
    (ResNet50,       'ResNet50'),
]

baseline_results = {}

for fn, name in baselines:
    print(f"\n🚀 {name}...")
    K.clear_session()
    gc.collect()

    model = build_baseline(fn, name)
    model.fit(
        X_train, y_train_oh,
        validation_data=(X_val, y_val_oh),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks(f'{name}_best.h5'),
        verbose=1
    )

    loss, acc, auc = model.evaluate(X_test, y_test_oh, verbose=0)
    y_pred  = np.argmax(model.predict(X_test, verbose=0), axis=1)
    report  = classification_report(
        y_test, y_pred, target_names=CLASS_NAMES, output_dict=True
    )
    baseline_results[name] = {
        'accuracy': acc * 100,
        'loss':     loss,
        'auc':      auc,
        'report':   report,
        'y_pred':   y_pred
    }
    print(f"  ✅ {name}: Acc={acc*100:.2f}% Loss={loss:.4f} AUC={auc:.4f}")

    del model
    gc.collect()


# ─────────────────────────────────────────────
# SECTION 11: TRAIN DRM-Net (proposed model)
# ─────────────────────────────────────────────

print("\n" + "="*60)
print("TRAINING DRM-Net (Proposed Model)")
print("="*60)

K.clear_session()
gc.collect()

drm_model = build_drm_net(input_shape=(CROP_SIZE, CROP_SIZE, 3))
drm_model.summary()

drm_model.compile(
    optimizer=Adam(LR),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print(f"\n🚀 Training DRM-Net  (epochs={EPOCHS}, bs={BATCH_SIZE}, lr={LR})")

drm_hist = drm_model.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks('drm_net_best.h5'),
    verbose=1
)


# ─────────────────────────────────────────────
# SECTION 12: EVALUATE DRM-Net
# Target: Acc=96.71%, Prec=96%, Recall=97%, F1=97%, AUC=99%
# ─────────────────────────────────────────────

print("\n" + "="*60)
print("DRM-Net EVALUATION")
print("="*60)

# Load best checkpoint
drm_model.load_weights('drm_net_best.h5')

loss_drm, acc_drm, auc_drm = drm_model.evaluate(X_test, y_test_oh, verbose=0)
y_pred_drm  = drm_model.predict(X_test, verbose=0)
y_cls_drm   = np.argmax(y_pred_drm, axis=1)

report_drm = classification_report(
    y_test, y_cls_drm, target_names=CLASS_NAMES, output_dict=True
)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_cls_drm, target_names=CLASS_NAMES))

print(f"\n🎯 DRM-Net Final Results:")
print(f"   Accuracy  : {acc_drm*100:.2f}%   (Paper: 96.71%)")
print(f"   Loss      : {loss_drm:.4f}       (Paper: ~0.15)")
print(f"   AUC       : {auc_drm*100:.2f}%   (Paper: 99%)")
print(f"   Precision : {report_drm['macro avg']['precision']*100:.1f}%   (Paper: 96%)")
print(f"   Recall    : {report_drm['macro avg']['recall']*100:.1f}%   (Paper: 97%)")
print(f"   F1-score  : {report_drm['macro avg']['f1-score']*100:.1f}%   (Paper: 97%)")


# ─────────────────────────────────────────────
# SECTION 13: CONFUSION MATRIX (Figure 8)
# ─────────────────────────────────────────────

def plot_cm(y_true, y_pred, title='Confusion Matrix with labels'):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='RdYlBu_r',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.5,
        annot_kws={'size': 16, 'weight': 'bold'}
    )
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Predicted', fontsize=13)
    ax.set_ylabel('Actual', fontsize=13)
    plt.tight_layout()
    plt.savefig('confusion_matrix_DRM_Net.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nCM:\n{cm}")

plot_cm(y_test, y_cls_drm)


# ─────────────────────────────────────────────
# SECTION 14: TRAINING CURVES (Figures 9–11)
# ─────────────────────────────────────────────

def plot_curves(history, name='DRM-Net'):
    h   = history.history
    ep  = range(1, len(h['accuracy']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{name} Training Curves', fontsize=16, fontweight='bold')

    for ax, tr_key, val_key, ylabel in [
        (axes[0], 'accuracy',     'val_accuracy',     'Accuracy'),
        (axes[1], 'loss',         'val_loss',          'Loss'),
        (axes[2], 'auc',          'val_auc',           'AUC'),
    ]:
        if tr_key not in h:
            continue
        ax.plot(ep, h[tr_key],  'r-o', label=f'Train {ylabel}',
                linewidth=2, markersize=4)
        ax.plot(ep, h[val_key], 'g-o', label=f'Val {ylabel}',
                linewidth=2, markersize=4)
        ax.set_title(f'Training and Validation {ylabel}', fontsize=13)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{name}_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_curves(drm_hist)


# ─────────────────────────────────────────────
# SECTION 15: FIVE-FOLD CROSS-VALIDATION (Table 7)
# Paper: 96.71% ± 0.2 over 5 folds
# ─────────────────────────────────────────────

print("\n" + "="*60)
print("5-FOLD CROSS-VALIDATION (Table 7)")
print("="*60)

skf          = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X_aug, y_aug), 1):
    print(f"\n  Fold {fold}/5...")
    K.clear_session()
    gc.collect()

    Xf_tr, Xf_te = X_aug[tr_idx], X_aug[te_idx]
    yf_tr, yf_te = y_aug[tr_idx], y_aug[te_idx]
    yf_tr_oh     = to_categorical(yf_tr, NUM_CLASSES)
    yf_te_oh     = to_categorical(yf_te, NUM_CLASSES)

    fm = build_drm_net(input_shape=(CROP_SIZE, CROP_SIZE, 3))
    fm.compile(
        optimizer=Adam(LR),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    fm.fit(
        Xf_tr, yf_tr_oh,
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            EarlyStopping(patience=8, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(factor=0.5, patience=4, verbose=0)
        ],
        verbose=0
    )

    _, af, aucf = fm.evaluate(Xf_te, yf_te_oh, verbose=0)
    y_pf = np.argmax(fm.predict(Xf_te, verbose=0), axis=1)
    rep  = classification_report(yf_te, y_pf, target_names=CLASS_NAMES,
                                 output_dict=True)

    fold_results.append({
        'Fold':      fold,
        'Accuracy':  round(af * 100, 2),
        'Precision': round(rep['macro avg']['precision'], 2),
        'Recall':    round(rep['macro avg']['recall'], 2),
        'F1-score':  round(rep['macro avg']['f1-score'], 2),
        'AUC':       round(aucf, 2)
    })
    print(f"    Fold {fold}: Acc={af*100:.2f}% Prec={rep['macro avg']['precision']:.2f} "
          f"Recall={rep['macro avg']['recall']:.2f} F1={rep['macro avg']['f1-score']:.2f} AUC={aucf:.2f}")

    del fm
    gc.collect()

fold_df = pd.DataFrame(fold_results)
print("\n📊 5-Fold CV Results (Paper Table 7):")
print(fold_df.to_string(index=False))
print(f"\n  Avg Accuracy: {fold_df['Accuracy'].mean():.2f} ± {fold_df['Accuracy'].std():.2f}")


# ─────────────────────────────────────────────
# SECTION 16: MODEL COMPARISON (Figures 12 & 13)
# ─────────────────────────────────────────────

def plot_comparison(baseline_res, drm_acc, drm_loss):
    names  = list(baseline_res.keys()) + ['DRM-Net']
    accs   = [baseline_res[m]['accuracy'] for m in baseline_res] + [drm_acc * 100]
    losses = [baseline_res[m]['loss']     for m in baseline_res] + [drm_loss]
    colors = ['#2ecc71','#3498db','#e67e22','#9b59b6','#e74c3c','#1abc9c','#f39c12']

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, vals, ylabel, ylim, title in [
        (axes[0], accs,   'Accuracy (%)', [70, 100], 'Accuracy Comparison (Testing)'),
        (axes[1], losses, 'Loss',          None,      'Loss Comparison (Testing)'),
    ]:
        bars = ax.bar(names, vals, color=colors, alpha=0.85,
                      edgecolor='white', linewidth=1.5)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=12)
        if ylim:
            ax.set_ylim(ylim)
        ax.tick_params(axis='x', rotation=30)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + (0.2 if ylim else 0.005),
                    f'{v:.2f}', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_comparison(baseline_results, acc_drm, loss_drm)


# ─────────────────────────────────────────────
# SECTION 17: CLASS-WISE METRICS TABLE (Table 8)
# ─────────────────────────────────────────────

all_results = {**baseline_results,
               'DRM-Net': {'report': report_drm, 'y_pred': y_cls_drm}}

print("\n📊 Class-wise Precision / Recall / F1 (Table 8):")
print(f"\n{'Model':<16} {'Class':<12} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 58)
for mn, res in all_results.items():
    rep = res['report']
    for i, cls in enumerate(CLASS_NAMES):
        p = rep[cls]['precision']
        r = rep[cls]['recall']
        f = rep[cls]['f1-score']
        prefix = mn if i == 0 else ''
        print(f"{prefix:<16} {cls:<12} {p:>10.2f} {r:>8.2f} {f:>8.2f}")
    print()


# ─────────────────────────────────────────────
# SECTION 18: XAI — GRAD-CAM VARIANTS
# Paper Section 7 / Figures 17–19
# ─────────────────────────────────────────────

def find_last_conv(model):
    for layer in reversed(model.layers):
        if isinstance(layer, Conv2D):
            return layer.name
    return None


def grad_cam(model, img, cls_idx, layer_name):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    img_t = tf.cast(img[np.newaxis], tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_t)
        loss = preds[:, cls_idx]
    grads   = tape.gradient(loss, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap  = conv_out @ pooled[..., tf.newaxis]
    heatmap  = tf.squeeze(heatmap)
    heatmap  = tf.maximum(heatmap, 0)
    heatmap  = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap(img_norm, heatmap, size):
    """Overlay jet colormap heatmap on de-normalized image"""
    disp = np.clip(img_norm * STD + MEAN, 0, 1)
    disp = (disp * 255).astype(np.uint8)
    hm   = cv2.resize(heatmap, (size[1], size[0]))
    hm   = np.uint8(255 * hm)
    hm   = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    hm   = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
    out  = cv2.addWeighted(disp, 0.6, hm, 0.4, 0)
    return out


def visualize_cam(model, X_samp, y_samp, cls_name, n=4):
    cls_idx  = CLASS_NAMES.index(cls_name)
    idxs     = np.where(y_samp == cls_idx)[0][:n]
    if len(idxs) == 0:
        return

    last_conv = find_last_conv(model)
    fig, axes = plt.subplots(2, len(idxs), figsize=(4*len(idxs), 8))
    fig.suptitle(
        f'Grad-CAM heatmaps — {cls_name} class (DRM-Net)',
        fontsize=14, fontweight='bold'
    )
    row_labels = ['Original', 'Grad-CAM']

    for col, idx in enumerate(idxs):
        img  = X_samp[idx]
        disp = np.clip(img * STD + MEAN, 0, 1)
        pred = np.argmax(model.predict(img[np.newaxis], verbose=0))

        axes[0, col].imshow(disp)
        axes[0, col].set_title(f'GT:{cls_name}\nPred:{CLASS_NAMES[pred]}',
                               fontsize=9)
        axes[0, col].axis('off')

        try:
            hm = grad_cam(model, img, pred, last_conv)
            ov = overlay_heatmap(img, hm, (CROP_SIZE, CROP_SIZE))
            axes[1, col].imshow(ov)
        except Exception as e:
            print(f"  CAM error: {e}")
        axes[1, col].axis('off')

    for row, lbl in enumerate(row_labels):
        axes[row, 0].set_ylabel(lbl, fontsize=11)

    plt.tight_layout()
    plt.savefig(f'CAM_{cls_name}.png', dpi=120, bbox_inches='tight')
    plt.show()


print("\n🎨 Generating Grad-CAM visualizations...")
for cls in CLASS_NAMES:
    visualize_cam(drm_model, X_test, y_test, cls, n=4)


# ─────────────────────────────────────────────
# SECTION 19: STATISTICAL ANALYSIS (Section 6)
# Standard deviation + Friedman aligned ranks test
# ─────────────────────────────────────────────

print("\n" + "="*60)
print("STATISTICAL ANALYSIS (Section 6)")
print("="*60)

print(f"\n{'Model':<16} {'Prec SD':>9} {'Recall SD':>10} {'F1 SD':>8}")
print("-" * 46)
for mn, res in all_results.items():
    rep   = res['report']
    precs = [rep[c]['precision'] for c in CLASS_NAMES]
    recs  = [rep[c]['recall']    for c in CLASS_NAMES]
    f1s   = [rep[c]['f1-score']  for c in CLASS_NAMES]
    print(f"{mn:<16} {np.std(precs):>9.3f} {np.std(recs):>10.3f} {np.std(f1s):>8.3f}")

# Friedman test on accuracies
accs_all  = [baseline_results[m]['accuracy'] for m in baseline_results]
accs_all += [acc_drm * 100]
stat, p   = stats.friedmanchisquare(*[np.array([v]) for v in accs_all])
print(f"\nFriedman statistic: {stat:.5f}")
print(f"p-value           : {p:.5f}")
print(f"H0                : {'Accepted (p > 0.05)' if p > 0.05 else 'Rejected'}")
print("(Paper: p=0.42319 → H0 Accepted)")


# ─────────────────────────────────────────────
# SECTION 20: ABLATION STUDY (Table 12)
# ─────────────────────────────────────────────

print("\n" + "="*60)
print("ABLATION STUDY (Table 12)")
print("="*60)

def build_mini_ensemble(configs, input_shape=(224, 224, 3)):
    """Lightweight version for ablation (frozen bases)"""
    inp = Input(shape=input_shape)
    branches = []
    for fn, nm in configs:
        base = fn(weights='imagenet', include_top=False,
                  input_shape=input_shape)
        base.trainable = False
        x = base(inp, training=False)
        x = GlobalAveragePooling2D()(x)
        x = Dense(256, activation='relu')(x)
        x = Dropout(DROPOUT)(x)
        branches.append(x)
    merged = Concatenate()(branches) if len(branches) > 1 else branches[0]
    merged = Dense(256, activation='relu')(merged)
    merged = Dropout(DROPOUT)(merged)
    out    = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(merged)
    model  = Model(inp, out)
    model.compile(optimizer=Adam(LR), loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model


ablation_configs = [
    ([(DenseNet169, 'dn')],                          'DenseNet169'),
    ([(ResNet50,    'rn')],                          'ResNet50'),
    ([(MobileNet,   'mn')],                          'MobileNet'),
    ([(DenseNet169, 'dn'), (ResNet50,  'rn')],       'DenseNet169+ResNet50'),
    ([(ResNet50,    'rn'), (MobileNet, 'mn')],       'ResNet50+MobileNet'),
    ([(DenseNet169, 'dn'), (MobileNet, 'mn')],       'DenseNet169+MobileNet'),
]
ablation_rows = []

for config, label in ablation_configs:
    K.clear_session(); gc.collect()
    print(f"\n  {label}...")
    am = build_mini_ensemble(config)
    am.fit(
        X_train, y_train_oh,
        validation_data=(X_val, y_val_oh),
        epochs=20, batch_size=BATCH_SIZE,
        callbacks=[EarlyStopping(patience=6, restore_best_weights=True,
                                 verbose=0)],
        verbose=0
    )
    _, aa = am.evaluate(X_test, y_test_oh, verbose=0)
    ypa  = np.argmax(am.predict(X_test, verbose=0), axis=1)
    rpa  = classification_report(y_test, ypa, target_names=CLASS_NAMES,
                                 output_dict=True)
    ablation_rows.append({
        'Model':     label,
        'Accuracy':  f"{aa*100:.2f}",
        'Precision': f"{rpa['macro avg']['precision']:.2f}",
        'Recall':    f"{rpa['macro avg']['recall']:.2f}",
        'F1-score':  f"{rpa['macro avg']['f1-score']:.2f}"
    })
    print(f"    {label}: {aa*100:.2f}%")
    del am; gc.collect()

# Add DRM-Net row
ablation_rows.append({
    'Model':     'DRM-Net (D+R+M)',
    'Accuracy':  f"{acc_drm*100:.2f}",
    'Precision': f"{report_drm['macro avg']['precision']:.2f}",
    'Recall':    f"{report_drm['macro avg']['recall']:.2f}",
    'F1-score':  f"{report_drm['macro avg']['f1-score']:.2f}"
})

abl_df = pd.DataFrame(ablation_rows)
print("\n📊 Ablation Results (Table 12):")
print(abl_df.to_string(index=False))


# ─────────────────────────────────────────────
# SECTION 21: MACRO & WEIGHTED COMPARISON (Figures 14–15)
# ─────────────────────────────────────────────

def plot_macro_weighted(all_res):
    names   = list(all_res.keys())
    metrics = ['precision', 'recall', 'f1-score']
    colors  = ['#2ecc71','#3498db','#e67e22','#9b59b6','#e74c3c','#1abc9c','#f39c12']

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    for ax_i, avg_type in enumerate(['macro avg', 'weighted avg']):
        ax = axes[ax_i]
        x  = np.arange(len(metrics))
        w  = 0.11
        for i, (mn, color) in enumerate(zip(names, colors)):
            rep  = all_res[mn]['report']
            vals = [rep[avg_type][m] for m in metrics]
            ax.bar(x + i * w, vals, w, label=mn,
                   color=color, alpha=0.85)
        ax.set_xticks(x + w * (len(names) - 1) / 2)
        ax.set_xticklabels(['Precision', 'Recall', 'F1-score'], fontsize=12)
        ax.set_ylim([0.75, 1.01])
        ax.set_title(
            ('Macro' if avg_type == 'macro avg' else 'Weighted') + ' Average',
            fontsize=14, fontweight='bold'
        )
        ax.set_ylabel('Score')
        ax.legend(loc='lower right', fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('macro_weighted_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_macro_weighted(all_results)


# ─────────────────────────────────────────────
# SECTION 22: SAVE MODEL & FINAL SUMMARY
# ─────────────────────────────────────────────

drm_model.save('drm_net_final.h5')
print("\n💾 Model saved: drm_net_final.h5")

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"""
╔═══════════════════════════════════════════════════════╗
║            DRM-Net PERFORMANCE SUMMARY               ║
╠═══════════════════════════════════════════════════════╣
║  Metric          │ Your Result    │ Paper Target     ║
╠═══════════════════════════════════════════════════════╣
║  Accuracy        │ {acc_drm*100:>6.2f}%        │ 96.71%           ║
║  Precision (mac) │ {report_drm['macro avg']['precision']:>6.2f}          │ 96%              ║
║  Recall (mac)    │ {report_drm['macro avg']['recall']:>6.2f}          │ 97%              ║
║  F1-Score (mac)  │ {report_drm['macro avg']['f1-score']:>6.2f}          │ 97%              ║
║  AUC             │ {auc_drm:>6.2f}          │ 99% (0.99)       ║
╠═══════════════════════════════════════════════════════╣
║  Dataset  : BUSI Kaggle — 2531 images (post-aug)    ║
║  Classes  : Benign / Malignant / Normal              ║
║  Split    : 70% train / 10% val / 20% test          ║
║  Backbone : DenseNet169 + ResNet50 + MobileNet       ║
║  Epochs   : {EPOCHS}  │ BatchSize: {BATCH_SIZE}  │ LR: {LR}      ║
╚═══════════════════════════════════════════════════════╝
""")

print("📁 Saved files:")
for f in [
    'confusion_matrix_DRM_Net.png',
    'DRM-Net_training_curves.png',
    'model_comparison.png',
    'macro_weighted_comparison.png',
    'CAM_benign.png', 'CAM_malignant.png', 'CAM_normal.png',
    'drm_net_best.h5', 'drm_net_final.h5'
]:
    print(f"   - {f}")

print("\n✅ All done!")

2026-04-29 04:06:03.426479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777435563.808663      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777435563.929644      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777435564.914847      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777435564.914892      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777435564.914895      57 computation_placer.cc:177] computation placer alr

✅ Libraries imported
TensorFlow: 2.19.0
GPUs available: 2
Mixed precision: float16 enabled

📁 Dataset: /kaggle/input/datasets/aryashah2k/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Contents: ['benign', 'normal', 'malignant']

📊 Loading dataset (takes ~5–10 min with masking)...
  benign: 437 images
  malignant: 210 images
  normal: 133 images

✅ Loaded: (780, 224, 224, 3)
   Benign=437, Malignant=210, Normal=133

🔄 Augmenting dataset...
  benign: 437 → 891 (+454)
  malignant: 210 → 842 (+632)
  normal: 133 → 798 (+665)

✅ After augmentation: (2531, 224, 224, 3)
   Benign=891, Malignant=842, Normal=798

📦 Split 70/10/20:
   Train=1770, Val=254, Test=507

TRAINING BASELINE TL MODELS

🚀 VGG16...


I0000 00:00:1777435651.485121      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777435651.490974      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/30


I0000 00:00:1777435658.648839     133 service.cc:152] XLA service 0x7ac1f4115ff0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777435658.648879     133 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777435658.648883     133 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777435659.383793     133 cuda_dnn.cc:529] Loaded cuDNN version 91002


...

2026-04-29 04:28:36.868966: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng11{k2=1,k3=0} for conv %cudnn-conv-bias-activation.174 = (f32[32,1024,7,7]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,1664,7,7]{3,2,1,0} %bitcast.89109, f32[1024,1664,7,7]{3,2,1,0} %bitcast.72257, f32[1024]{0} %bitcast.89179), window={size=7x7 pad=3_3x3_3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv2D" op_name="DRM_Net_1/dn_conv_1/convolution" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-04-29 04:28:36.907752: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.038962377s
Trying alg

56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4330 - auc: 0.6291 - loss: 1.2706   

2026-04-29 04:31:56.610830: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng11{k2=4,k3=0} for conv %cudnn-conv-bias-activation.168 = (f32[32,1024,7,7]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,1664,7,7]{3,2,1,0} %bitcast.34841, f32[1024,1664,7,7]{3,2,1,0} %bitcast.34848, f32[1024]{0} %bitcast.34850), window={size=7x7 pad=3_3x3_3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv2D" op_name="DRM_Net_1/dn_conv_1/convolution" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kRelu","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-04-29 04:31:57.570218: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.959551172s
Trying alg

56/56 ━━━━━━━━━━━━━━━━━━━━ 336s 4s/step - accuracy: 0.4341 - auc: 0.6300 - loss: 1.2694 - val_accuracy: 0.3819 - val_auc: 0.5405 - val_loss: 12.3119 - learning_rate: 0.0010
Epoch 2/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.6073 - auc: 0.7900 - loss: 0.9331 - val_accuracy: 0.3701 - val_auc: 0.5297 - val_loss: 24.7208 - learning_rate: 0.0010
Epoch 3/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - accuracy: 0.7490 - auc: 0.8973 - loss: 0.6471 - val_accuracy: 0.3346 - val_auc: 0.5094 - val_loss: 17.5510 - learning_rate: 0.0010
Epoch 4/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 942ms/step - accuracy: 0.7940 - auc: 0.9213 - loss: 0.6083

56/56 ━━━━━━━━━━━━━━━━━━━━ 70s 1s/step - accuracy: 0.7941 - auc: 0.9214 - loss: 0.6074 - val_accuracy: 0.4921 - val_auc: 0.6274 - val_loss: 5.5557 - learning_rate: 0.0010
Epoch 5/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.8520 - auc: 0.9587 - loss: 0.3952 - val_accuracy: 0.4528 - val_auc: 0.6733 - val_loss: 2.6346 - learning_rate: 0.0010
Epoch 6/30
 7/56 ━━━━━━━━━━━━━━━━━━━━ 45s 936ms/step - accuracy: 0.9077 - auc: 0.9811 - loss: 0.2672